# k01 — Engineering pilot (STAGE2_DESIGN_FROZEN_v1.0, §10)
set_01/train_01 only, 2 recording-level folds, one seed. Measures extraction time, epoch time, memory, capacity matching, ID-permutation invariance, rewiring change fraction, checkpoint reload, LightGBM/rule timing, and projects total GPU-hours. **No test cell is extracted or scored.** Validation AP values are sanity checks only, not results.

In [ ]:
import os
os.makedirs('/kaggle/working/pilot_code', exist_ok=True)
FILES = {'feats.py': "import numpy as np, pandas as pd, re, os, time\nW = 64\nHEXV = np.full(256, 0, dtype=np.uint8)\nfor i, ch in enumerate('0123456789abcdef'):\n    HEXV[ord(ch)] = i; HEXV[ord(ch.upper())] = i\nPOP = np.array([bin(i).count('1') for i in range(256)], dtype=np.uint8)\n\ndef slog(x):\n    return np.sign(x) * np.log1p(np.abs(x))\n\ndef load_file(path):\n    df = pd.read_csv(path, dtype={'arbitration_id': str, 'data_field': str, 'attack': np.int8}, keep_default_na=True)\n    ts = df['timestamp'].to_numpy(np.float64)\n    ids = df['arbitration_id'].str.rjust(3, '0').str[-3:]\n    ib = np.frombuffer(''.join(ids.tolist()).encode('ascii'), dtype=np.uint8).reshape(-1, 3)\n    idv = HEXV[ib].astype(np.int32)\n    id_int = idv[:, 0] * 256 + idv[:, 1] * 16 + idv[:, 2]\n    d = df['data_field'].fillna('')\n    plen = (d.str.len().to_numpy() // 2).clip(0, 8).astype(np.int8)\n    d16 = d.str[:16].str.ljust(16, '0')\n    db = np.frombuffer(''.join(d16.tolist()).encode('ascii'), dtype=np.uint8).reshape(-1, 16)\n    nib = HEXV[db]\n    pay = (nib[:, 0::2] * 16 + nib[:, 1::2]).astype(np.uint8)\n    posmask = np.arange(8)[None, :] < plen[:, None]\n    pay = np.where(posmask, pay, 0).astype(np.uint8)\n    y = df['attack'].to_numpy(np.int8)\n    return ts, id_int, plen, pay, y\n\ndef per_frame_globals(ts, id_int, plen, pay):\n    n = len(ts); idx = np.arange(n)\n    order = np.lexsort((idx, id_int))\n    prev = np.full(n, -1, dtype=np.int64)\n    same = np.r_[False, id_int[order][1:] == id_int[order][:-1]]\n    prev[order[same]] = order[np.flatnonzero(same) - 1]\n    has = prev >= 0\n    pp = np.where(has, prev, 0)\n    dt_same = np.where(has, ts - ts[pp], 0.0)\n    x = pay ^ pay[pp]\n    ham = np.where(has, POP[x].sum(1), 0).astype(np.float32)\n    maxlen = np.maximum(plen, plen[pp]).astype(np.float32)\n    chg = np.where(has, (x != 0).sum(1) / np.maximum(maxlen, 1), 0).astype(np.float32)\n    lenchg = np.where(has, plen != plen[pp], False)\n    ent = np.zeros(n, dtype=np.float32)\n    for s in range(0, n, 500000):\n        b = pay[s:s + 500000]; L = plen[s:s + 500000].astype(np.int32)\n        valid = np.arange(8)[None, :] < L[:, None]\n        eq = (b[:, :, None] == b[:, None, :]) & valid[:, :, None] & valid[:, None, :]\n        c = eq.sum(2).astype(np.float32)\n        Lf = np.maximum(L, 1).astype(np.float32)[:, None]\n        with np.errstate(divide='ignore', invalid='ignore'):\n            term = np.where(valid, np.log2(np.where(c > 0, c, 1) / Lf), 0.0)\n        ent[s:s + 500000] = -(term.sum(1) / Lf[:, 0])\n    return prev, dt_same, ham, chg, ent, lenchg\n\nFRAME_NAMES = ['plen', 'dt_prev_any', 'dt_same', 'no_prev_same_in_window', 'hamming', 'changed_frac', 'entropy', 'same_as_prev']\nNODE_NAMES = ['count', 'first_pos', 'last_pos', 'ia_mean', 'ia_min', 'ia_max', 'ia_missing', 'plen_mean', 'plen_max', 'plen_changes', 'ham_mean', 'chg_mean', 'ent_mean']\nGLOBAL_NAMES = ['duration', 'distinct_ids', 'distinct_transitions', 'fps']\n\ndef windows_for_file(path, stride, id_perm=None):\n    ts, id_int, plen, pay, y = load_file(path)\n    if id_perm is not None:\n        id_int = id_perm[id_int]\n    n = len(ts)\n    if n < W:\n        return None\n    prev, dt_same_g, ham_g, chg_g, ent_g, lenchg_g = per_frame_globals(ts, id_int, plen, pay)\n    starts = np.arange(0, n - W + 1, stride)\n    nw = len(starts)\n    I = starts[:, None] + np.arange(W)[None, :]\n    ok = prev[I] >= starts[:, None]\n    tsw = ts[I]\n    dtp = np.diff(tsw, axis=1, prepend=tsw[:, :1])\n    idw = id_int[I]\n    fr = np.zeros((nw, W, len(FRAME_NAMES)), dtype=np.float32)\n    fr[..., 0] = plen[I] / 8.0\n    fr[..., 1] = slog(dtp * 1000)\n    fr[..., 2] = np.where(ok, slog(dt_same_g[I] * 1000), 0)\n    fr[..., 3] = ~ok\n    fr[..., 4] = np.where(ok, ham_g[I] / 64.0, 0)\n    fr[..., 5] = np.where(ok, chg_g[I], 0)\n    fr[..., 6] = ent_g[I] / 3.0\n    fr[:, 1:, 7] = idw[:, 1:] == idw[:, :-1]\n    # nodes\n    key = (np.arange(nw)[:, None] * 4096 + idw).ravel()\n    uk, inv = np.unique(key, return_inverse=True)\n    inv = inv.reshape(nw, W)\n    win_of_node = uk // 4096\n    node_first = np.searchsorted(win_of_node, np.arange(nw))\n    local = inv - node_first[:, None]\n    nn = np.bincount(win_of_node, minlength=nw)\n    G = len(uk)\n    fl = inv.ravel()\n    pos = np.broadcast_to(np.arange(W), (nw, W)).ravel()\n    okf = ok.ravel()\n    def agg_sum(v, m=None):\n        return np.bincount(fl, weights=(v if m is None else v * m), minlength=G)\n    order = np.argsort(fl, kind='stable'); fs = fl[order]\n    bnd = np.flatnonzero(np.r_[True, fs[1:] != fs[:-1]])\n    def agg_min(v): return np.minimum.reduceat(v[order], bnd)\n    def agg_max(v): return np.maximum.reduceat(v[order], bnd)\n    cnt = np.bincount(fl, minlength=G).astype(np.float32)\n    iak = np.where(okf, slog(dt_same_g[I].ravel() * 1000), np.nan)\n    nia = agg_sum(okf.astype(np.float64))\n    has_ia = nia > 0\n    ia_mean = np.where(has_ia, agg_sum(np.nan_to_num(iak)) / np.maximum(nia, 1), 0)\n    ia_min = np.where(has_ia, agg_min(np.where(okf, iak, np.inf)), 0)\n    ia_max = np.where(has_ia, agg_max(np.where(okf, iak, -np.inf)), 0)\n    pl = (plen[I].ravel()).astype(np.float64)\n    nd = np.zeros((G, len(NODE_NAMES)), dtype=np.float32)\n    nd[:, 0] = cnt / W\n    nd[:, 1] = agg_min(pos.astype(np.float64)) / W\n    nd[:, 2] = agg_max(pos.astype(np.float64)) / W\n    nd[:, 3] = ia_mean; nd[:, 4] = ia_min; nd[:, 5] = ia_max\n    nd[:, 6] = ~has_ia\n    nd[:, 7] = agg_sum(pl) / cnt / 8.0\n    nd[:, 8] = agg_max(pl) / 8.0\n    nd[:, 9] = agg_max(pl) != agg_min(pl)\n    nd[:, 10] = np.where(has_ia, agg_sum(ham_g[I].ravel().astype(np.float64), okf) / np.maximum(nia, 1) / 64.0, 0)\n    nd[:, 11] = np.where(has_ia, agg_sum(chg_g[I].ravel().astype(np.float64), okf) / np.maximum(nia, 1), 0)\n    nd[:, 12] = agg_sum(ent_g[I].ravel().astype(np.float64)) / cnt / 3.0\n    node = np.zeros((nw, W, len(NODE_NAMES)), dtype=np.float32)\n    node[win_of_node, np.arange(G) - node_first[win_of_node]] = nd\n    # edges: local src/dst per transition\n    src = local[:, :-1].astype(np.uint8); dst = local[:, 1:].astype(np.uint8)\n    tr = np.unique((np.arange(nw)[:, None] * 4096 + local[:, :-1] * 64 + local[:, 1:]).ravel())\n    ntr = np.bincount(tr // 4096, minlength=nw)\n    dur = tsw[:, -1] - tsw[:, 0]\n    glob = np.stack([slog(dur * 1000), nn / W, ntr / 63.0, slog(W / np.maximum(dur, 1e-6))], 1).astype(np.float32)\n    lab = (y[I].max(1) > 0).astype(np.int8)\n    nattack = y[I].sum(1).astype(np.int16)\n    return dict(frame=fr.astype(np.float16), node=node.astype(np.float16), nmask=(np.arange(W)[None, :] < nn[:, None]),\n                src=src, dst=dst, glob=glob, y=lab, nattack=nattack, starts=starts.astype(np.int64), t0=tsw[:, 0], t1=tsw[:, -1])\n", 'models.py': "import torch, torch.nn as nn, numpy as np, time\nW = 64\n\ndef mlp(i, h, o, drop):\n    return nn.Sequential(nn.Linear(i, h), nn.ReLU(), nn.Dropout(drop), nn.Linear(h, o))\n\nclass Head(nn.Module):\n    def __init__(self, h, g, drop):\n        super().__init__(); self.rho = mlp(3 * h + g, h, 1, drop)\n    def forward(self, H, mask, glob):\n        m = mask.unsqueeze(-1).float()\n        s = (H * m).sum(1); mean = s / m.sum(1).clamp(min=1)\n        mx = H.masked_fill(m == 0, -1e4).max(1).values\n        return self.rho(torch.cat([s, mean, mx, glob], 1)).squeeze(-1)\n\nclass DeepSets(nn.Module):\n    def __init__(self, f, h, g, drop=0.0):\n        super().__init__()\n        self.phi = nn.Sequential(nn.Linear(f, h), nn.ReLU(), nn.Dropout(drop), nn.Linear(h, h), nn.ReLU())\n        self.head = Head(h, g, drop)\n    def forward(self, b):\n        return self.head(self.phi(b['node']), b['nmask'], b['glob'])\n\nclass SAGELayer(nn.Module):\n    def __init__(self, i, o):\n        super().__init__(); self.self_lin = nn.Linear(i, o); self.nei_lin = nn.Linear(i, o, bias=False)\n    def forward(self, H, A):\n        # A[b, src, dst] = weight; aggregate incoming neighbours of each dst node (weighted mean)\n        agg = torch.bmm(A.transpose(1, 2), H)\n        deg = A.sum(1).unsqueeze(-1)\n        agg = agg / deg.clamp(min=1e-9)\n        return self.self_lin(H) + self.nei_lin(agg)\n\nclass GraphSAGE(nn.Module):\n    def __init__(self, f, h, g, drop=0.0):\n        super().__init__()\n        self.l1 = SAGELayer(f, h); self.l2 = SAGELayer(h, h); self.drop = nn.Dropout(drop)\n        self.head = Head(h, g, drop)\n    def forward(self, b):\n        A = b['adj']; m = b['nmask'].unsqueeze(-1).float()\n        H = torch.relu(self.l1(b['node'], A)) * m\n        H = torch.relu(self.l2(self.drop(H), A)) * m\n        return self.head(H, b['nmask'], b['glob'])\n\nclass GRUNet(nn.Module):\n    def __init__(self, f, h, g, drop=0.0):\n        super().__init__()\n        self.gru = nn.GRU(f, h, batch_first=True); self.out = mlp(2 * h + g, h, 1, drop)\n    def forward(self, b):\n        O, hn = self.gru(b['frame'])\n        return self.out(torch.cat([hn[-1], O.mean(1), b['glob']], 1)).squeeze(-1)\n\ndef nparams(m):\n    return sum(p.numel() for p in m.parameters())\n\ndef build_adj(src, dst, B, device):\n    A = torch.zeros(B, W, W, device=device)\n    bi = torch.arange(B, device=device).unsqueeze(1).expand_as(src)\n    A.index_put_((bi.reshape(-1), src.reshape(-1).long(), dst.reshape(-1).long()), torch.full((src.numel(),), 1.0 / 63, device=device), accumulate=True)\n    return A\n\ndef rewire_dst(dst, gen):\n    # degree-preserving directed rewiring: permute destination endpoints among a window's 63 edges\n    # (every source keeps its out-degree, every destination keeps its in-degree, multiplicities included)\n    r = torch.rand(dst.shape, generator=gen, device=dst.device)\n    perm = r.argsort(1)\n    return torch.gather(dst, 1, perm)\n\ndef edge_change_fraction(src, dst, dst2):\n    # fraction of the 63 directed edges (as a multiset per window) not present in the original\n    B = src.shape[0]\n    k1 = (src.long() * 64 + dst.long()).sort(1).values\n    k2 = (src.long() * 64 + dst2.long()).sort(1).values\n    fr = []\n    for i in range(B):\n        a, ca = torch.unique(k1[i], return_counts=True); b2, cb = torch.unique(k2[i], return_counts=True)\n        common = 0\n        d = dict(zip(a.tolist(), ca.tolist()))\n        for kk, cc in zip(b2.tolist(), cb.tolist()):\n            common += min(cc, d.get(kk, 0))\n        fr.append(1 - common / 63.0)\n    return float(np.mean(fr))\n", 'set01_train_hashes.py': "SET01_TRAIN = [('set_01/train_01/accessory-1.csv', '8085578', '01fd35561a0756782d05b8aa99af1eb555ad3dfa51137a90e05901adf826007e'), ('set_01/train_01/accessory-2.csv', '8768770', '6cab63f1d04a06cade61b21b356f26524095fc7bc10de39a86ba9fee2ca5b5cb'), ('set_01/train_01/attack-free-1.csv', '75976161', 'ac377ea04cfcc0c742761ff9e6fad2ba80ab9bbeefffc0ff8fdabd0be9c2f05c'), ('set_01/train_01/attack-free-2.csv', '49385649', 'a78ed1e6e3c2ddd6711584a7806acbdf146ae4c04cc15a2caf4fae83c755671a'), ('set_01/train_01/DoS-1.csv', '3498722', '70ba91332949ec1ce68faa5ff89e9647b9e271153108ccc5eff4c24d7e10117d'), ('set_01/train_01/DoS-2.csv', '12122150', 'ed28da2b77dc3c556ebe54f73f6c4aec8dcf9a744b9b306ad858951f01f85809'), ('set_01/train_01/force-neutral-1.csv', '27754807', '90f42168f45241ea6979849b421405a28bab9556701a9ca1133bc6e6ea464c89'), ('set_01/train_01/force-neutral-2.csv', '38541656', 'a84b2a1d2ca680a29cd8c212c6bd983f4a82d58d1cde2ff4c0cf1c182db37717'), ('set_01/train_01/rpm-1.csv', '32695456', '44e6b912666f4e089b885f910d2f9211c6faf99c74dc87d4f79cac33a86a3b31'), ('set_01/train_01/rpm-2.csv', '31718456', '6627e889f6bc0a7d60e20a632ab7f645a758d4a3af69e0525b540b71303c6a56'), ('set_01/train_01/standstill-1.csv', '75978651', '11a514b6d9a1007acef158dcd21fc48f168e2f1024221b3f28bcf13baca7867f'), ('set_01/train_01/standstill-2.csv', '48937495', '29629eba417975566943220ecc87fb562e05c730094d0f3dd1f38c2fa105b5a7')]\n", 'pilot_main.py': "import os, sys, json, time, glob, hashlib, subprocess, traceback, math, re\nimport numpy as np, torch\nfrom sklearn.metrics import average_precision_score\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nimport feats, models\nfrom set01_train_hashes import SET01_TRAIN\n\nOUT = '/kaggle/working'\nREPORT = {'stage': 'pilot', 'design': 'STAGE2_DESIGN_FROZEN_v1.0', 'note': 'engineering pilot: set_01/train_01 only, 2 recording-level folds, no test cell read', 'errors': []}\ndef save():\n    json.dump(REPORT, open(OUT + '/pilot_report.json', 'w'), indent=1, default=str)\ndef log(k, v):\n    REPORT[k] = v; print(k, json.dumps(v, default=str)[:2000], flush=True); save()\ndef stage(name, fn):\n    t = time.time()\n    try:\n        r = fn(); log('time_' + name + '_s', round(time.time() - t, 1)); return r\n    except Exception as e:\n        REPORT['errors'].append({'stage': name, 'error': repr(e), 'trace': traceback.format_exc()[-3000:]}); save(); print('ERROR in', name, e, flush=True); return None\n\ndev = 'cuda' if torch.cuda.is_available() else 'cpu'\nenv = {'device': dev, 'torch': torch.__version__}\nif dev == 'cuda':\n    env['gpu'] = torch.cuda.get_device_name(0); env['n_gpu'] = torch.cuda.device_count()\nenv['ram_GB'] = round(os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 2**30, 1); env['cpus'] = os.cpu_count()\nlog('env', env)\n\nZIP = '/kaggle/temp/can.zip'; ROOT = '/kaggle/temp/data/can-train-and-test'\ndef get_data():\n    os.makedirs('/kaggle/temp', exist_ok=True)\n    if not os.path.exists(ZIP) or os.path.getsize(ZIP) != 1507455719:\n        subprocess.run(['wget', '-q', '-O', ZIP, 'https://ndownloader.figshare.com/files/43632393'], check=True)\n    h = hashlib.md5()\n    with open(ZIP, 'rb') as f:\n        for b in iter(lambda: f.read(8 << 20), b''): h.update(b)\n    assert h.hexdigest() == 'bd6509d670c0a0009cb3ecab34111bcd', 'zip md5 mismatch'\n    # extract ONLY set_01/train_01 (test cells are never extracted in the pilot)\n    subprocess.run(['unzip', '-q', '-o', ZIP, 'can-train-and-test/set_01/train_01/*', '-d', '/kaggle/temp/data'], check=True)\n    extracted = sorted(glob.glob(ROOT + '/**/*.csv', recursive=True))\n    assert all('/train_01/' in p for p in extracted), 'non-train file extracted'\n    bad = []\n    for rel, size, sha in SET01_TRAIN:\n        p = ROOT + '/' + rel; hh = hashlib.sha256()\n        with open(p, 'rb') as f:\n            for b in iter(lambda: f.read(8 << 20), b''): hh.update(b)\n        if hh.hexdigest() != sha or os.path.getsize(p) != int(size): bad.append(rel)\n    assert not bad, 'hash mismatch ' + str(bad)\n    return {'n_extracted': len(extracted), 'all_train_only': True, 'sha256_ok': True}\nr = stage('data', get_data); log('data', r)\nif r is None: sys.exit(0)\n\nfiles = sorted(glob.glob(ROOT + '/set_01/train_01/*.csv'))\nfam = lambda p: re.sub(r'-\\d+\\.csv$', '', os.path.basename(p))\nidx = lambda p: int(re.search(r'-(\\d+)\\.csv$', p).group(1))\nlo = min(idx(p) for p in files)\nfolds = {1: ([p for p in files if idx(p) == lo], [p for p in files if idx(p) != lo]),\n         2: ([p for p in files if idx(p) != lo], [p for p in files if idx(p) == lo])}\nlog('folds', {k: {'train': [os.path.basename(p) for p in v[0]], 'val': [os.path.basename(p) for p in v[1]]} for k, v in folds.items()})\n\nFEAT = {}\ndef extract():\n    stats = {}\n    for p in files:\n        t = time.time(); d = feats.windows_for_file(p, 32); FEAT[p] = d\n        stats[os.path.basename(p)] = {'windows_s32': int(len(d['y'])), 'pos_s32': int(d['y'].sum()), 's': round(time.time() - t, 1)}\n    return stats\nlog('extraction', stage('extract', extract))\n\ndef idperm_test():\n    p = files[0]; rng = np.random.default_rng(1); perm = rng.permutation(4096)\n    a = FEAT[p]; b = feats.windows_for_file(p, 32, id_perm=perm)\n    ok = np.array_equal(a['frame'], b['frame']) and np.array_equal(a['glob'], b['glob']) and np.array_equal(a['y'], b['y'])\n    for w in rng.choice(len(a['y']), 500, replace=False):\n        x = a['node'][w][a['nmask'][w]]; z = b['node'][w][b['nmask'][w]]\n        ok &= np.array_equal(x[np.lexsort(x.T[::-1])], z[np.lexsort(z.T[::-1])])\n    return bool(ok)\nlog('id_permutation_feature_invariance', stage('idperm', idperm_test))\n\ndef cat(plist, stride64):\n    keys = ['frame', 'node', 'nmask', 'src', 'dst', 'glob', 'y']\n    out = {k: [] for k in keys}\n    for p in plist:\n        d = FEAT[p]; sel = (d['starts'] % 64 == 0) if stride64 else slice(None)\n        for k in keys: out[k].append(d[k][sel])\n    return {k: np.concatenate(v) for k, v in out.items()}\n\ndef to_batch(D, ix, rewire=False, gen=None, norm=None):\n    b = {}\n    b['frame'] = torch.from_numpy(D['frame'][ix].astype(np.float32)).to(dev)\n    b['node'] = torch.from_numpy(D['node'][ix].astype(np.float32)).to(dev)\n    b['nmask'] = torch.from_numpy(D['nmask'][ix]).to(dev)\n    b['glob'] = torch.from_numpy(D['glob'][ix]).to(dev)\n    if norm is not None:\n        b['frame'] = (b['frame'] - norm['fm']) / norm['fs']; b['node'] = (b['node'] - norm['nm']) / norm['ns'] * b['nmask'].unsqueeze(-1); b['glob'] = (b['glob'] - norm['gm']) / norm['gs']\n    src = torch.from_numpy(D['src'][ix]).to(dev); dst = torch.from_numpy(D['dst'][ix]).to(dev)\n    if rewire: dst = models.rewire_dst(dst, gen)\n    b['adj'] = models.build_adj(src, dst, len(ix), dev)\n    return b\n\ndef norm_stats(D):\n    f = D['frame'].astype(np.float32).reshape(-1, D['frame'].shape[-1])\n    m = D['nmask'].reshape(-1); n = D['node'].astype(np.float32).reshape(-1, D['node'].shape[-1])[m]\n    T = lambda x: torch.tensor(x, dtype=torch.float32, device=dev)\n    return {'fm': T(f.mean(0)), 'fs': T(f.std(0) + 1e-6), 'nm': T(n.mean(0)), 'ns': T(n.std(0) + 1e-6), 'gm': T(D['glob'].mean(0)), 'gs': T(D['glob'].std(0) + 1e-6)}\n\nNF, NN, NG = len(feats.FRAME_NAMES), len(feats.NODE_NAMES), len(feats.GLOBAL_NAMES)\ndef make(name, h):\n    if name == 'GraphSAGE' or name == 'GraphSAGE_rewired': return models.GraphSAGE(NN, h, NG)\n    if name == 'DeepSets': return models.DeepSets(NN, h, NG)\n    if name == 'GRU': return models.GRUNet(NF, h, NG)\n\ndef match_hidden():\n    target = models.nparams(make('GraphSAGE', 64)); res = {'GraphSAGE': (64, target)}\n    for nm in ['DeepSets', 'GRU']:\n        best = min(range(8, 257), key=lambda h: abs(models.nparams(make(nm, h)) - target))\n        res[nm] = (best, models.nparams(make(nm, best)))\n    res['GraphSAGE_rewired'] = res['GraphSAGE']\n    return {k: {'hidden': v[0], 'params': v[1], 'ratio_to_graphsage': round(v[1] / target, 3)} for k, v in res.items()}\nHID = stage('capacity', match_hidden); log('capacity', HID)\n\nEPOCHS = int(os.environ.get('PILOT_EPOCHS', '4')); BS = 1024\ndef train_eval(name, fold):\n    torch.manual_seed(0); gen = torch.Generator(device=dev); gen.manual_seed(0)\n    Dtr = cat(folds[fold][0], False); Dva = cat(folds[fold][1], True)\n    norm = norm_stats(Dtr)\n    m = make(name, HID[name]['hidden']).to(dev)\n    opt = torch.optim.AdamW(m.parameters(), lr=1e-3)\n    pos = Dtr['y'].sum(); neg = len(Dtr['y']) - pos\n    lossf = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(neg / max(pos, 1), device=dev))\n    rew = name == 'GraphSAGE_rewired'\n    rec = {'n_train_windows': int(len(Dtr['y'])), 'n_train_pos': int(pos), 'n_val_windows': int(len(Dva['y'])), 'n_val_pos': int(Dva['y'].sum()), 'epochs': []}\n    if dev == 'cuda': torch.cuda.reset_peak_memory_stats()\n    for ep in range(EPOCHS):\n        m.train(); t = time.time(); perm = np.random.default_rng(ep).permutation(len(Dtr['y'])); tot = 0\n        for s in range(0, len(perm), BS):\n            ix = np.sort(perm[s:s + BS]); b = to_batch(Dtr, ix, rew, gen, norm)\n            yb = torch.from_numpy(Dtr['y'][ix].astype(np.float32)).to(dev)\n            loss = lossf(m(b), yb); opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item() * len(ix)\n        if dev == 'cuda': torch.cuda.synchronize()\n        tr_s = time.time() - t\n        m.eval(); t = time.time(); sc = []\n        g2 = torch.Generator(device=dev); g2.manual_seed(12345)\n        with torch.no_grad():\n            for s in range(0, len(Dva['y']), 4096):\n                ix = np.arange(s, min(s + 4096, len(Dva['y']))); sc.append(m(to_batch(Dva, ix, rew, g2, norm)).float().cpu().numpy())\n        sc = np.concatenate(sc); ev_s = time.time() - t\n        ap = average_precision_score(Dva['y'], sc) if Dva['y'].sum() > 0 else None\n        rec['epochs'].append({'epoch': ep, 'train_loss': round(tot / len(perm), 5), 'train_s': round(tr_s, 1), 'val_score_s': round(ev_s, 1), 'val_ap_sanity': None if ap is None else round(float(ap), 4)})\n        print(name, fold, rec['epochs'][-1], flush=True)\n    if dev == 'cuda': rec['peak_gpu_mem_MB'] = round(torch.cuda.max_memory_allocated() / 2**20)\n    ck = OUT + f'/ckpt_{name}_f{fold}.pt'; torch.save({'model': m.state_dict(), 'opt': opt.state_dict(), 'epoch': EPOCHS}, ck)\n    m2 = make(name, HID[name]['hidden']).to(dev); st = torch.load(ck, map_location=dev); m2.load_state_dict(st['model'])\n    rec['checkpoint_reload_identical'] = all(torch.equal(a, b) for a, b in zip(m.state_dict().values(), m2.state_dict().values()))\n    os.remove(ck)\n    return rec\n\nRES = {}\nfor name in ['DeepSets', 'GraphSAGE', 'GraphSAGE_rewired', 'GRU']:\n    for fold in [1, 2]:\n        r = stage(f'train_{name}_f{fold}', lambda: train_eval(name, fold))\n        RES[f'{name}_f{fold}'] = r; log('neural', RES)\n\ndef rewiring():\n    D = cat(folds[1][1], True); ix = np.random.default_rng(0).choice(len(D['y']), 2000, replace=False)\n    src = torch.from_numpy(D['src'][ix]).to(dev); dst = torch.from_numpy(D['dst'][ix]).to(dev)\n    g = torch.Generator(device=dev); g.manual_seed(0)\n    return {'mean_edge_change_fraction': round(models.edge_change_fraction(src, dst, models.rewire_dst(dst, g)), 4), 'n_windows': 2000,\n            'rule': 'F3: if < 0.50 use directed configuration-model randomisation'}\nlog('rewiring', stage('rewiring', rewiring))\n\ndef lgbm_and_rule():\n    import lightgbm as lgb\n    def flat(D):\n        m = D['nmask'][..., None]; x = D['node'].astype(np.float32); cnt = m.sum(1).clip(1)\n        mean = (x * m).sum(1) / cnt; std = np.sqrt(((x - mean[:, None]) ** 2 * m).sum(1) / cnt)\n        mn = np.where(m, x, np.inf).min(1); mx = np.where(m, x, -np.inf).max(1)\n        return np.concatenate([mean, std, mn, mx, D['glob']], 1)\n    out = {}\n    for fold in [1, 2]:\n        Dtr = cat(folds[fold][0], False); Dva = cat(folds[fold][1], True)\n        Xtr, Xva = flat(Dtr), flat(Dva); pos = Dtr['y'].sum(); neg = len(Dtr['y']) - pos\n        t = time.time()\n        clf = lgb.LGBMClassifier(n_estimators=300, num_leaves=31, learning_rate=0.1, min_child_samples=20, scale_pos_weight=neg / max(pos, 1), verbose=-1)\n        clf.fit(Xtr, Dtr['y']); fit_s = time.time() - t\n        ap = average_precision_score(Dva['y'], clf.predict_proba(Xva)[:, 1])\n        t = time.time(); best = (-1, None, None)\n        for j in range(Xva.shape[1]):\n            for sgn in (1, -1):\n                a = average_precision_score(Dva['y'], sgn * Xva[:, j])\n                if a > best[0]: best = (a, j, sgn)\n        out[f'f{fold}'] = {'lgbm_fit_s': round(fit_s, 1), 'lgbm_val_ap_sanity': round(float(ap), 4), 'n_features': int(Xtr.shape[1]), 'rule_select_s': round(time.time() - t, 1), 'rule_feature_index': best[1], 'rule_sign': best[2], 'rule_val_ap_sanity': round(float(best[0]), 4)}\n    return out\nlog('lgbm_rule', stage('lgbm_rule', lgbm_and_rule))\n\ndef projection():\n    # windows at stride 32 per set's train_01 (from Stage 1C non-overlapping counts x2)\n    tw = {'set_01': 332898, 'set_02': 541886, 'set_03': 375792, 'set_04': 296630}\n    per_win = {}\n    for name in ['DeepSets', 'GraphSAGE', 'GraphSAGE_rewired', 'GRU']:\n        e = [RES[f'{name}_f{f}'] for f in (1, 2) if RES.get(f'{name}_f{f}')]\n        if not e: continue\n        s = np.mean([np.mean([x['train_s'] + x['val_score_s'] for x in r['epochs'][1:]] or [r['epochs'][0]['train_s']]) / r['n_train_windows'] for r in e])\n        per_win[name] = s\n    EPOCH_CAP = 20; hours = {}\n    for name, s in per_win.items():\n        full = sum(tw.values()) * s * EPOCH_CAP / 3600       # one full-train run on all 4 sets\n        half = full / 2                                      # one fold run (half the recordings)\n        tuning = 0 if name == 'GraphSAGE_rewired' else 8 * 2 * half\n        finals = 5 * full; ablation = 5 * full\n        sens = 2 * 5 * full                                  # W=32,128 (evaluation-set restriction not modelled)\n        hours[name] = {'tuning_h': round(tuning, 2), 'finals_h': round(finals, 2), 'd4_ablation_h': round(ablation, 2), 'window_sensitivity_h': round(sens, 2), 'total_h': round(tuning + finals + ablation + sens, 2)}\n    return {'assumed_epoch_cap': EPOCH_CAP, 'per_model': hours, 'total_gpu_hours_all_models': round(sum(v['total_h'] for v in hours.values()), 1)}\nlog('projection', stage('projection', projection))\nlog('status', 'DONE')\n"}
for n, s in FILES.items():
    open('/kaggle/working/pilot_code/' + n, 'w').write(s)
print(sorted(os.listdir('/kaggle/working/pilot_code')))

In [ ]:
!PILOT_EPOCHS=4 python /kaggle/working/pilot_code/pilot_main.py

In [ ]:
import json
print(json.dumps(json.load(open('/kaggle/working/pilot_report.json')), indent=1)[:20000])